<div dir="rtl" style="text-align:right"><h1>שיעור 3, חלק 2: איך מודל שפה כותב – שלב A: המודל מקבל רצף יחידות</h1>
<p><b>המחברת הזו היא לשלב A («המודל מקבל רצף יחידות») בשיעור באתר הקורס – ורק לו.</b> את ההסבר קוראים באתר; כאן מריצים ומשנים.</p>
<p>מריצים תא אחרי תא בלחיצה על ▶, מלמעלה למטה. מעבד רגיל (<code>CPU</code>) מספיק; אין צורך ב-<code>GPU</code>, במפתח <code>API</code> או במנוי. <b>אין חובת הגשת מחברת.</b> אם מופיעה אזהרה על <code>HF_TOKEN</code> – אפשר להתעלם ממנה: המודל ציבורי.</p></div>

In [ ]:
# Setup: tiktoken is the open-source tokenizer library OpenAI publishes.
# transformers is already installed on Google Colab.
%pip install -q tiktoken

import tiktoken
import transformers
from transformers import AutoTokenizer

print("tiktoken", tiktoken.__version__, "| transformers", transformers.__version__, "- setup done")

<div dir="rtl" style="text-align:right"><h2>1 | טוקנים: מה המודל מקבל בפועל</h2>
<p>מודל שפה לא מקבל אותיות או מילים, אלא רצף של <b>טוקנים</b>: חתיכות טקסט מתוך אוצר קבוע. את החיתוך עושה <b>tokenizer</b>. טוקן יכול להיות מילה שלמה, חלק ממילה, אות אחת – ולפעמים אפילו חצי אות (חלק מהבתים שמייצגים אות במחשב).</p>
<p>נבדוק שלושה <code>tokenizer</code>-ים אמיתיים, כל אחד בשמו:</p>
<ul>
<li><code>o200k_base</code> – הקידוד ש-OpenAI פרסמה בספריית <code>tiktoken</code> ומשמש חלק מהמודלים החדשים שלה.</li>
<li><code>cl100k_base</code> – קידוד ותיק יותר מאותה ספרייה.</li>
<li><code>Qwen2.5</code> – ה-<code>tokenizer</code> של המודל הפתוח שנשתמש בו בשלב B.</li>
</ul>
<p><b>חשוב:</b> אלה מדידות אמיתיות של שלושת ה-<code>tokenizer</code>-ים האלה בלבד. הצ׳טבוט שאתם משתמשים בו עשוי להשתמש ב-<code>tokenizer</code> אחר, ולכן המספרים אצלו יכולים להיות שונים.</p>
<blockquote>① לפני ההרצה: נחשו – למשפט «מיכל רוצה שלוש עוגיות למחר» יהיו יותר טוקנים או פחות מאשר לתרגום שלו לאנגלית?</blockquote></div>

In [ ]:
TOKENIZERS = {
    "o200k_base": tiktoken.get_encoding("o200k_base"),
    "cl100k_base": tiktoken.get_encoding("cl100k_base"),
    "Qwen2.5": AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B"),
}


def pieces(name, text):
    """The token pieces of `text` under one tokenizer. A piece that is only
    part of a letter's bytes cannot be shown on its own and prints as �."""
    tok = TOKENIZERS[name]
    if name == "Qwen2.5":
        ids = tok.encode(text)
        return [tok.decode([i]) for i in ids]
    ids = tok.encode(text)
    return [tok.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in ids]


def show_tokens(text):
    print("Text:", text)
    for name in TOKENIZERS:
        parts = pieces(name, text)
        print(f"  {name:12} {len(parts):3} tokens:  " + " | ".join(p.replace(" ", "·") for p in parts))
    print()


show_tokens("מיכל רוצה שלוש עוגיות למחר")
show_tokens("Michal wants three cookies for tomorrow")
print("· marks a space that belongs to the token.  � marks part of a letter (a piece of its bytes).")

<div dir="rtl" style="text-align:right"><h3>1א | אותו תוכן, שתי שפות</h3>
<p>התא הבא סופר טוקנים לכמה זוגות של משפטים עם אותו תוכן בעברית ובאנגלית. שימו לב לעמודת היחס: כמה טוקנים בעברית על כל טוקן באנגלית.</p></div>

In [ ]:
PAIRS = [
    ("מיכל רוצה שלוש עוגיות למחר", "Michal wants three cookies for tomorrow"),
    ("שעת האיסוף עדיין לא סוכמה", "The pickup time has not been agreed yet"),
    ("בבוקר, לפני העבודה, אני שותה כוס קפה", "In the morning, before work, I drink a cup of coffee"),
]

print(f"{'tokenizer':12} {'Hebrew':>7} {'English':>8} {'ratio':>6}")
for name in TOKENIZERS:
    he = sum(len(pieces(name, h)) for h, _ in PAIRS)
    en = sum(len(pieces(name, e)) for _, e in PAIRS)
    print(f"{name:12} {he:7} {en:8} {he / en:6.2f}")

<div dir="rtl" style="text-align:right"><h3>1ב | התור שלכם</h3>
<p>③ כתבו בין המירכאות משפט משלכם – למשל הודעה קצרה של לקוח – והריצו. נסו גם מילה עם תחילית («ובבית», «כשהגעתי»), מספר («3») ושם פרטי.</p>
<blockquote>שאלות לחשיבה (לא לבדיקה): איפה ה-<code>tokenizer</code> חתך מילה לחלקים? מה קורה למספר? למה זה משנה – לכמות הטקסט שמודל מקבל בבת אחת (<b>חלון ההקשר</b>) יש גבול שנמדד בטוקנים, ובשירותים בתשלום המחיר נקבע לרוב לפי מספר הטוקנים.</blockquote></div>

In [ ]:
my_text = "דנה: למחר בשעה 09:00, ארבע לחמניות ושתי עוגיות."  # <- change this text and run again
show_tokens(my_text)

<div dir="rtl" style="text-align:right"><p><b>סיימתם את המחברת של שלב A.</b> חזרו ללשונית של אתר הקורס וענו שם על שאלות השלב.</p></div>